# Compare dbr_cb_adjust experiments

Load per-run metrics from all `*dbr_cb_adjust*` simulation experiments, build comparison tables (metrics × experiments), and plot boxplots for selected metrics.

In [ ]:
from pathlib import Path
import re
import time

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display

from manusim.metrics import ExperimentMetrics

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENTS_ROOT = PROJECT_ROOT / "data" / "experiments"
NAME_FILTER = "dbr_cb_adjust"

sns.set_theme(style="whitegrid")

## Discover experiments

In [ ]:
def read_experiment_config(experiment_path: Path) -> dict:
    cfg_path = experiment_path / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        return {}
    return yaml.safe_load(cfg_path.read_text(encoding="utf-8"))


def has_run_metrics(path: Path) -> bool:
    if (path / "runs_metrics.csv").is_file():
        return True
    return any(path.glob("run_*/metrics_products.csv"))


def discover_experiments(root: Path, name_filter: str) -> list[Path]:
    candidates = sorted(
        p for p in root.rglob("*")
        if p.is_dir() and name_filter in p.name
    )
    experiments = []
    for path in candidates:
        if not (path / ".hydra" / "config.yaml").exists():
            continue
        if not (any(path.rglob("log_*.csv")) or has_run_metrics(path)):
            continue
        experiments.append(path)
    return experiments


def build_registry(experiment_paths: list[Path]) -> pd.DataFrame:
    rows = []
    for path in experiment_paths:
        config = read_experiment_config(path)
        sim_cfg = config.get("simulation", {}) or {}
        cb_target_level = sim_cfg.get("cb_target_level")
        label = f"cb{cb_target_level}" if cb_target_level is not None else path.name
        rows.append(
            {
                "folder": path.name,
                "path": str(path),
                "label": label,
                "cb_target_level": cb_target_level,
                "buffers_update_multiplyer": sim_cfg.get("buffers_update_multiplyer"),
                "buffers_update_warmup": sim_cfg.get("buffers_update_warmup"),
                "cb_update": sim_cfg.get("cb_update"),
                "sb_update": sim_cfg.get("sb_update"),
            }
        )
    registry = pd.DataFrame(rows)
    if not registry.empty:
        registry = registry.sort_values(["cb_target_level", "folder"], na_position="last")
    return registry.reset_index(drop=True)


experiment_paths = discover_experiments(EXPERIMENTS_ROOT, NAME_FILTER)
registry = build_registry(experiment_paths)

print(f"Found {len(registry)} experiment(s) matching '{NAME_FILTER}'")
display(registry)

## Load runs metrics

Loads pre-aggregated per-run metrics via `ExperimentMetrics.read_runs_metrics()` (from `runs_metrics.csv` or `run_*/metrics_*.csv`), without reading time-series logs.

In [ ]:
def load_runs_metrics(experiment_path: Path) -> pd.DataFrame:
    metrics = ExperimentMetrics(experiment_path)
    return metrics.read_runs_metrics()


label_by_folder = dict(zip(registry["folder"], registry["label"]))
column_order = registry["label"].tolist()

runs_metrics_parts = []
total = len(experiment_paths)
t_all = time.perf_counter()

for i, path in enumerate(experiment_paths, start=1):
    label = label_by_folder.get(path.name, path.name)
    t0 = time.perf_counter()
    print(f"{i}/{total}: {label} ({path.name}) ...", end=" ", flush=True)
    runs_df = load_runs_metrics(path)
    runs_df["experiment_label"] = label
    runs_metrics_parts.append(runs_df)
    print(f"done in {time.perf_counter() - t0:.1f}s ({len(runs_df)} rows)")

all_runs_metrics = pd.concat(runs_metrics_parts, ignore_index=True)
print(f"\nTotal load time: {time.perf_counter() - t_all:.1f}s")
print(f"Combined shape: {all_runs_metrics.shape}")
all_runs_metrics.head()

## Metric names

In [ ]:
def make_metric_name(row) -> str:
    key = str(row["key"]).replace("produto", "_p").replace("general", "_general")
    key = re.sub(r"^r(\d{2})", r"_r\1", key)
    return row["variable"] + key


all_runs_metrics = all_runs_metrics.copy()
all_runs_metrics["metric"] = all_runs_metrics.apply(make_metric_name, axis=1)

print("Standard variables:", sorted(all_runs_metrics["variable"].unique()))
print("Detailed metrics:", len(all_runs_metrics["metric"].unique()))

## Aggregated comparison table

Rows = standard metrics (mean across keys, then mean across runs). Columns = experiments ordered by `cb_target_level`.

In [ ]:
run_level_agg = (
    all_runs_metrics.groupby(["experiment_label", "variable", "run"], dropna=False)["value"]
    .mean()
    .reset_index()
)

agg_comparison = (
    run_level_agg.groupby(["variable", "experiment_label"])["value"]
    .mean()
    .unstack("experiment_label")
)

agg_comparison = agg_comparison.reindex(columns=column_order)
agg_comparison.index.name = "metric"
display(agg_comparison)

## Detailed comparison table

Rows = variable + key (e.g. `lostSales_p01`). Columns = experiments.

In [ ]:
detailed_comparison = (
    all_runs_metrics.groupby(["metric", "experiment_label"])["value"]
    .mean()
    .unstack("experiment_label")
)

detailed_comparison = detailed_comparison.reindex(columns=column_order)
detailed_comparison.index.name = "metric"
display(detailed_comparison)

## Boxplot: metric distribution across experiments

Per-run values are shown as boxplots so you can compare spread and outliers, not just means.

In [ ]:
SELECTED_METRIC = "lostSales"  # standard variable name
AGGREGATE_KEYS = True          # True: mean across keys per run; False: use SELECTED_KEY
SELECTED_KEY = "general"       # used when AGGREGATE_KEYS=False

available_variables = sorted(all_runs_metrics["variable"].unique())
available_metrics = sorted(all_runs_metrics["metric"].unique())
print("Available variables:", available_variables)
print(f"Total detailed metrics: {len(available_metrics)}")

In [ ]:
plot_df = all_runs_metrics.loc[all_runs_metrics["variable"] == SELECTED_METRIC].copy()

if plot_df.empty:
    raise ValueError(f"No data for variable '{SELECTED_METRIC}'. Choose from: {available_variables}")

if AGGREGATE_KEYS:
    plot_df = (
        plot_df.groupby(["experiment_label", "run"], as_index=False)["value"]
        .mean()
    )
    title_suffix = "(mean across keys per run)"
else:
    plot_df = plot_df.loc[plot_df["key"] == SELECTED_KEY]
    title_suffix = f"(key={SELECTED_KEY})"

if plot_df.empty:
    raise ValueError(
        f"No data for variable '{SELECTED_METRIC}' with key '{SELECTED_KEY}'."
    )

fig, ax = plt.subplots(figsize=(max(8, len(column_order) * 1.2), 5))
sns.boxplot(
    data=plot_df,
    x="experiment_label",
    y="value",
    order=column_order,
    ax=ax,
)
ax.set_xlabel("Experiment (cb_target_level)")
ax.set_ylabel(SELECTED_METRIC)
ax.set_title(f"{SELECTED_METRIC} across dbr_cb_adjust experiments {title_suffix}")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
detailed_comparison.T[['utilization_r09']]

In [ ]:
# Boxplot of resource9 usage using detailed_comparison
metric_r9 = "utilization_r09"  # typical naming for detailed metric of resource 9 usage

fig, ax = plt.subplots(figsize=(max(8, len(column_order) * 1.2), 5))
sns.boxplot(
    data=detailed_comparison.T.reset_index(),
    x="experiment_label",
    y="utilization_r09",
    order=column_order,
    ax=ax,
)
ax.set_xlabel("Experiment (cb_target_level)")
ax.set_ylabel("Resource 9 Utilization")
ax.set_title("Utilization of Resource 9 across dbr_cb_adjust experiments")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()